In [1]:
import pandas as pd

In [2]:
# Años de interés
years = [f"{y} " for y in range(2008, 2025)]

# Función genérica
def load_and_transform(file_path, id_col_names, value_name, filter_unit=None):
    # Leer archivo
    df = pd.read_csv(file_path, sep="\t", compression="gzip")

    # Separar la primera columna compuesta
    split_cols = id_col_names.split(',')
    df[split_cols] = df.iloc[:, 0].str.split(",", expand=True)
    df = df.drop(columns=df.columns[0])

    # Transformar a formato largo
    df_long = df.melt(id_vars=split_cols, value_vars=[col for col in df.columns if col in years],
                      var_name="year", value_name=value_name)
    df_long["year"] = df_long["year"].str.strip().astype(int)

    # Filtrar unidad si se requiere
    if filter_unit:
        df_long = df_long[df_long["unit"].str.contains(filter_unit, case=False, na=False)]

    # Renombrar país
    if "geo" in df_long.columns:
        df_long = df_long.rename(columns={"geo": "country"})

    # Conservar columnas finales
    df_clean = df_long[["country", "year", value_name]].copy()
    df_clean[value_name] = pd.to_numeric(df_clean[value_name], errors="coerce")

    return df_clean

In [7]:
# GDP
gdp_file = r"C:\Users\santi\OneDrive\Documentos\VSC\TFG\estat_nama_10_gdp.tsv.gz"
df_gdp = load_and_transform(gdp_file, "freq,unit,na_item,geo", "gdp", filter_unit="millones de euros")

# Debt

debt_file = r"C:\Users\santi\OneDrive\Documentos\VSC\TFG\estat_gov_10dd_edpt1.tsv.gz"
df_debt = load_and_transform(debt_file, "freq,unit,sector,na_item,geo", "debt", filter_unit="millones de euros")

# Inflation

inflation_file = r"C:\Users\santi\OneDrive\Documentos\VSC\TFG\estat_prc_hicp_aind.tsv.gz"
df_inflation = load_and_transform(inflation_file, "freq,unit,coicop,geo", "inflation")

# GINI

gini_file = r"C:\Users\santi\OneDrive\Documentos\VSC\TFG\estat_tessi190.tsv.gz"
df_gini = load_and_transform(gini_file, "freq,age,statinfo,geo", "gini")

# Employment

employment_file = r"C:\Users\santi\OneDrive\Documentos\VSC\TFG\estat_lfsi_emp_a.tsv.gz"
df_employment = load_and_transform(employment_file, "freq,indic_em,sex,age,unit,geo", "employment")

# Unemployment

unemployment_file = r"C:\Users\santi\OneDrive\Documentos\VSC\TFG\estat_une_rt_a.tsv.gz"
df_unemployment = load_and_transform(unemployment_file, "freq,age,unit,sex,geo", "unemployment")


In [8]:
from functools import reduce

dfs = [df_gdp, df_debt, df_inflation, df_gini, df_employment, df_unemployment]
df_merged = reduce(lambda left, right: pd.merge(left, right, on=["country", "year"], how="outer"), dfs)


MemoryError: Unable to allocate 16.4 GiB for an array with shape (2196264884,) and data type int64

In [9]:
for df in dfs:
    print(df.columns)
    print(df[["country", "year"]].drop_duplicates().sort_values(by=["country", "year"]).head())

Index(['country', 'year', 'gdp'], dtype='object')
Empty DataFrame
Columns: [country, year]
Index: []
Index(['country', 'year', 'debt'], dtype='object')
Empty DataFrame
Columns: [country, year]
Index: []
Index(['country', 'year', 'inflation'], dtype='object')
       country  year
29          AL  2008
35280       AL  2009
70531       AL  2010
105782      AL  2011
141033      AL  2012
Index(['country', 'year', 'gini'], dtype='object')
    country  year
0        AL  2014
38       AL  2015
76       AL  2016
114      AL  2017
152      AL  2018
Index(['country', 'year', 'employment'], dtype='object')
      country  year
0          AT  2008
2664       AT  2009
5328       AT  2010
7992       AT  2011
10656      AT  2012
Index(['country', 'year', 'unemployment'], dtype='object')
     country  year
0         AT  2008
2331      AT  2009
4662      AT  2010
6993      AT  2011
9324      AT  2012
